In [110]:
!pip install sqlalchemy
!pip install pymysql

8194.63s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
8201.12s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [111]:
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
from pathlib import Path
import re
import os

# Database
from sqlalchemy import Column, Integer, BigInteger, Float, String, Boolean, DateTime, Date, ForeignKey
from sqlalchemy.orm import declarative_base, relationship
from sqlalchemy import create_all_engines, create_engine
from sqlalchemy import URL

ImportError: cannot import name 'create_all_engines' from 'sqlalchemy' (/opt/conda/lib/python3.11/site-packages/sqlalchemy/__init__.py)

Це було написано для того щоб перевіряти скільки унікальних комбінацій колонок є в усіх таблиціях.
- Якщо 1 — всі колонки одінакові з однаковим порядком, можна спокійно мерджити
- Якщо ні — треба розбиратися

In [48]:
data_dir = "data/"

c_set = set()
files = Path(data_dir).rglob('*.csv')
for f in files:
    sdf = pd.read_csv(f)
    c_set.add(tuple(sdf.columns))

len(c_set)

1

Як видно всі колонки однакові тому дані можна обробляти до спільного формату. Спочаку об'єднаємо всі файли в один великий датафрейм для подальшої обробки

In [49]:
files = Path(data_dir).rglob('*.csv')
df = pd.concat(map(pd.read_csv, files))

In [50]:
df.head()

,stations_id,stations_name,Lat,Long,stations_time,stations_offset,stations_params_id,stations_params_key,stations_params_name,stations_params_localName,stations_params_unit,stations_params_localUnit,stations_params_value,stations_params_cr,stations_params_time,stations_params_offset,stations_params_level
0,767,Соборна 36,49.232859,28.470453,2022-10-25 13:24:28,0,33,SDS_P2,PM2.5,Пил 2.5 мкм,ug/m3,мкг/м³,9.14,1.000000,2022-10-25 13:24:28,0,1
1,767,Соборна 36,49.232859,28.470453,2022-10-25 13:24:28,0,34,SDS_P1,PM10,Пил 10 мкм,ug/m3,мкг/м³,11.69,1.000000,2022-10-25 13:24:28,0,1
2,767,Соборна 36,49.232859,28.470453,2022-10-25 13:24:28,0,7,CO2,CO₂,CO₂,ppm,мкг/м³,400.00,1.800009,2022-10-25 13:24:28,0,1
3,774,Станція замостя,49.245473,28.493727,2022-10-25 13:25:11,0,33,SDS_P2,PM2.5,Пил 2.5 мкм,ug/m3,мкг/м³,14.04,1.000000,2022-10-25 13:25:11,0,2
4,774,Станція замостя,49.245473,28.493727,2022-10-25 13:25:11,0,34,SDS_P1,PM10,Пил 10 мкм,ug/m3,мкг/м³,19.68,1.000000,2022-10-25 13:25:11,0,1


# Звередння даних що розходяться

**Якщо** один і той самий id має різні імена станцій вони виводяться

In [64]:
ids = df["stations_id"].unique()

for id in ids:
    names = df.loc[df["stations_id"] == id, "stations_name"].unique()
    if len(names) > 1:
        print(id, names)

На основі отриманих перетинів створимо мапінг щоб усунути однакові імена для різних станцій

In [63]:
stations_mapping = {
    256: "vinnytsia-256",
    281: "vinnytsia-281",
    315: "vinnytsia-315",
    90: "vinnytsia-90",
    271: "vinnytsia-271",
    767: "Соборна 36",
    1183: "Вишенька",
    246: "vinnytsia-246",
    274: "vinnytsia-274",
}

In [53]:
mask = df['stations_id'].isin(stations_mapping.keys())
df.loc[mask, 'stations_name'] = df.loc[mask, 'stations_id'].map(stations_mapping)

Перевірка на однакові значення.
- `True` означає що дані не перетинаються і можна йти далі
- `False` означає що дані перетинаються і потрібні додаткові правки

In [54]:
len(df['stations_id'].unique()) == len(df['stations_name'].unique())

True

Перетини усунуто, можна йти далі. Далі нормалізуємо координати.

**Кооридинати повинні відповідати наступним критеріям:**
1. одна станція має мати один набір координат
2. всі станції повинні мати довжину і широту

Перевіримо наявність перетину даних координат алгоритмом подібним до того що використовувався для імен станцій

In [62]:
ids = df["stations_id"].unique()  # Перелік унікальних станці

for id in ids:
    lats = df.loc[df["stations_id"] == id, "Lat"].unique()  # отримання унікальних широт
    longs = df.loc[df["stations_id"] == id, "Long"].unique()  # отримання унікальних довгот

    # Підрахунок кількості унікальних коодинат довгот і широт
    n_lats = len(lats)
    n_longs = len(longs)

    # Якщо спільні широти чи довготи вивести розділювач
    if n_lats != 1 or n_longs != 1:
        print("="*40)
        print(id)

    # Вивести дані що перетинаються
    if len(lats) > 1:
        print(f"спільні широти: {lats}")
    if len(longs) > 1:
        print(f"спільні довготи: {longs}")

Як можна побачити основна проблема полягає в тому що частина даних зберігається як текстове значення, а інша як числові значення. Між числовими значеннями наявна невелика відмнність у точності через що дані відрізняються та забруднюють датасет.

Спочатку приведемо дані до спільного формату, конвертуючи `str`в `int` та зводячи дані до однакової кількості чисел після коми

In [56]:
def process_coords(coord) -> float:
    PRESSISION: int = 5  # кількість чисел після коми

    # Випадок коли дані мають числове значення та їх потрібно перетворити в int
    if isinstance(coord, str):
        better_coord = re.match('\d+.\d+', coord).group()
        return round(float(better_coord), PRESSISION)

    # Випадок коли число просто слід скруглити
    if isinstance(coord, float):
        return round(coord, PRESSISION)

Застосуємо функцію для обробки до колонок

In [57]:
df["Lat"] = df["Lat"].apply(process_coords)
df["Long"] = df["Long"].apply(process_coords)

До оброблених даних застосуємо маппінг, щоб координати які зустрічаються найчастіше стали координатами станцій

In [58]:
# Мапінг широт
mode_per_id_lat = df.groupby('stations_id')['Lat'].agg(lambda x: x.mode().iloc[0])

# Мапінг довгот
mode_per_id_long = df.groupby('stations_id')['Long'].agg(lambda x: x.mode().iloc[0])

Застосуємо мапінг

In [66]:
# Застосування мапінгу
df['Lat'] = df['stations_id'].map(mode_per_id_lat)
df['Long'] = df['stations_id'].map(mode_per_id_long)

In [67]:
df.head()

,stations_id,stations_name,Lat,Long,stations_time,stations_offset,stations_params_id,stations_params_key,stations_params_name,stations_params_localName,stations_params_unit,stations_params_localUnit,stations_params_value,stations_params_cr,stations_params_time,stations_params_offset,stations_params_level
0,767,Соборна 36,49.23286,28.47045,2022-10-25 13:24:28,0,33,SDS_P2,PM2.5,Пил 2.5 мкм,ug/m3,мкг/м³,9.14,1.000000,2022-10-25 13:24:28,0,1
1,767,Соборна 36,49.23286,28.47045,2022-10-25 13:24:28,0,34,SDS_P1,PM10,Пил 10 мкм,ug/m3,мкг/м³,11.69,1.000000,2022-10-25 13:24:28,0,1
2,767,Соборна 36,49.23286,28.47045,2022-10-25 13:24:28,0,7,CO2,CO₂,CO₂,ppm,мкг/м³,400.00,1.800009,2022-10-25 13:24:28,0,1
3,774,Станція замостя,49.24547,28.49373,2022-10-25 13:25:11,0,33,SDS_P2,PM2.5,Пил 2.5 мкм,ug/m3,мкг/м³,14.04,1.000000,2022-10-25 13:25:11,0,2
4,774,Станція замостя,49.24547,28.49373,2022-10-25 13:25:11,0,34,SDS_P1,PM10,Пил 10 мкм,ug/m3,мкг/м³,19.68,1.000000,2022-10-25 13:25:11,0,1


# Підготовка бази даних

## Задання схеми даних

Розглянемо перелік колонок з якими ми будемо працювати.

In [78]:
df.dtypes

stations_id                    int64
stations_name                 object
Lat                          float64
Long                         float64
stations_time                 object
stations_offset                int64
stations_params_id            object
stations_params_key           object
stations_params_name          object
stations_params_localName     object
stations_params_unit          object
stations_params_localUnit     object
stations_params_value        float64
stations_params_cr           float64
stations_params_time          object
stations_params_offset         int64
stations_params_level          int64
dtype: object

Як видно з оброблених даних — більшість об'єктів залишаються типос object, тобто текстові дані. Для того щоб краще працювати з ними слід створити базу даних, використовуючи стандартну схему Data Warehouse, а саме – модель "Сніжинка".

В даному випадку будуть настунпі таблиці:
- **Фактологічна таблиця**
    - Виміри — поєднує решту таблиць бази даних
- **Вимірні таблиці**
    - Параметри – зберігає змінні параметри вимірів, такі як відхилення
    - Станції — зберігає дані про станцію та її властивості
    - Час – зберігає передоброблені часові дані
    - Одиниці виміру — зберігає дані про одиниці виміру

![](./uml/schema.png)

## Реалізаціія схеми даних використовуючи SQLalchemy

In [90]:
Base = declarative_base()

### Заданя вимірної таблиці вимірів

In [91]:
class DimUnit(Base):
    """
    Таблиця одиниць виміру, вимірна таблиця
    """
    __tablename__ = 'dim_units'

    unit_key = Column(Integer, primary_key=True, autoincrement=True)
    unit_name = Column(String, nullable=False)
    unit_symbol = Column(String)
    local_unit_name = Column(String)

    # Зв'язки
    parameters = relationship("DimParameter", back_populates="unit")

### Здааня таблиць параметрів

In [92]:
class DimParameter(Base):
    """
    Таблиця параметрів вимірів, вимірна таблиця
    """
    __tablename__ = 'dim_parameters'

    parameter_key = Column(Integer, primary_key=True, autoincrement=True)
    parameter_code = Column(String, nullable=False)
    parameter_name = Column(String)
    local_name = Column(String)
    unit_key = Column(Integer, ForeignKey('dim_units.unit_key'))

    # SCD Type 2 колонки
    valid_from = Column(DateTime, nullable=False)
    valid_to = Column(DateTime)
    is_current = Column(Boolean, default=True)

    # Зв'язки
    unit = relationship("DimUnit", back_populates="parameters")
    measurements = relationship("FactMeasurement", back_populates="parameter")

### Задання таблиць танцій

In [93]:
class DimStation(Base):
    """
    Таблиця станцій, вимірна таблиця
    """
    __tablename__ = 'dim_stations'

    station_key = Column(Integer, primary_key=True, autoincrement=True)
    station_id = Column(Integer)
    station_name = Column(String)
    latitude = Column(Float)
    longitude = Column(Float)
    timezone_offset = Column(Integer)

    # SCD Type 2 колонки
    valid_from = Column(DateTime, nullable=False)
    valid_to = Column(DateTime)
    is_current = Column(Boolean, default=True)

    # Зв'язки
    measurements = relationship("FactMeasurement", back_populates="station")

### Задання таблиць з часом

In [94]:
class DimTime(Base):
    """
    Таблиця деталей про час, вимірна таблиця
    """    
    __tablename__ = 'dim_time'

    time_key = Column(BigInteger, primary_key=True)
    full_timestamp = Column(DateTime, nullable=False)
    date = Column(Date, nullable=False)
    hour = Column(Integer)
    day = Column(Integer)
    day_of_week = Column(Integer)
    month = Column(Integer)
    quarter = Column(Integer)
    year = Column(Integer)

    # Зв'язки
    measurements = relationship("FactMeasurement", back_populates="time")

### Задання фактологічної таблиці

In [95]:
class FactMeasurement(Base):
    """
    Фактологічна таблиця
    """
    __tablename__ = 'fact_measurements'

    measurement_id = Column(BigInteger, primary_key=True, autoincrement=True)

    # Зовнішні ключи
    station_key = Column(Integer, ForeignKey('dim_stations.station_key'), nullable=False)
    parameter_key = Column(Integer, ForeignKey('dim_parameters.parameter_key'), nullable=False)
    time_key = Column(BigInteger, ForeignKey('dim_time.time_key'), nullable=False)

    # Виміри
    value = Column(Float)
    quality_ratio = Column(Float)
    pollution_level = Column(Integer)

    # Метаданні
    offset_minutes = Column(Integer)
    raw_timestamp = Column(DateTime)

    # Зв'язки
    station = relationship("DimStation", back_populates="measurements")
    parameter = relationship("DimParameter", back_populates="measurements")
    time = relationship("DimTime", back_populates="measurements")

## Створення фізичної моделі

In [113]:
engine = create_engine(
    URL.create(
        drivername="mysql",
        username= os.getenv("MYSQL_USER"),
        password= os.getenv("MYSQL_PASSWORD"),
        host="db",
        port=3306,
        database=os.getenv("MYSQL_DATABASE")
    )
)

NameError: name 'create_engine' is not defined

In [112]:
print(engine)

NameError: name 'engine' is not defined